In [1]:
import pandas as pd
df=pd.read_csv('synthetic_logs.csv')
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
df.log_message.sample(10).tolist()

['User 7153 made multiple incorrect login attempts',
 'System reboot initiated by user User648.',
 'System reboot initiated by user User876.',
 'Account with ID 4995 created by User559.',
 'nova.osapi_compute.wsgi.server [req-e584ef57-6eeb-4444-bd9a-e1bf5ea3149b 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 1893 time: 0.2322080',
 'System reboot initiated by user User938.',
 'nova.metadata.wsgi.server [req-e840cb3c-2e87-4b1c-8d42-7817a506acd7 - - - - -] 10.11.21.135,10.11.10.1 "GET /openstack/2013-10-17 HTTP/1.1" Return code: 200 len: 157 time: 0.2177148',
 'Critical system unit failure: unit ID Component76',
 'nova.osapi_compute.wsgi.server [req-60c3da91-bd1c-4693-991e-fa3580fa22c2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" Return code: 200 len: 1

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   timestamp     2410 non-null   object
 1   source        2410 non-null   object
 2   log_message   2410 non-null   object
 3   target_label  2410 non-null   object
 4   complexity    2410 non-null   object
dtypes: object(5)
memory usage: 94.3+ KB


In [5]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import numpy as np
model=SentenceTransformer('all-MiniLM-L6-v2',local_files_only=True)
embeddings=model.encode(df['log_message'].tolist())
#dbscan=DBSCAN(eps=0.5, min_samples

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embeddings[:3]

array([[-0.10293962,  0.03354593, -0.02202606, ...,  0.00457791,
        -0.04259717,  0.00322621],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296]], shape=(3, 384), dtype=float32)

In [8]:
dbscan= DBSCAN(eps=0.2,min_samples=1, metric='cosine')
clusters= dbscan.fit_predict(embeddings)
df['clusters']=clusters
df.head()

,timestamp,source,log_message,target_label,complexity,clusters
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [9]:
cluster_count=df.clusters.value_counts()
large_cluster=cluster_count[cluster_count>10].index
large_cluster

Index([ 0,  5, 11, 13,  7,  8, 21,  3,  4, 17,  6, 32, 16, 20,  9,  1, 10, 34,
       53, 14, 52, 18, 42, 25, 59, 26],
      dtype='int64', name='clusters')

In [10]:
for cluster in large_cluster:
    print(f'cluster {cluster}')
    print(df[df.clusters==cluster]['log_message'].head(5).to_string(index=False))
    print('/n')

cluster 0
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...
/n
cluster 5
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...
/n
cluster 11
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.
/n
cluster 13
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.
/n
cluster 7
Multiple bad login attempts detected on user 85...
Multiple login failures occurred on user 9052 a...
  Us

In [11]:
df[df.clusters==5]['log_message'].head(1).to_string(index=False)

'nova.compute.claims [req-a07ac654-8e81-416d-bfb...'

In [12]:
import re
def classify_with_regex(log_message):
    regex_patterns={
        r'User User\d+ logged (in|out).': 'User Action',
        r'Backup (started|ended) at .*': 'System Notification',
        r'Backup completed successfully.': 'System Notification',
        r'System updated to version .*': 'System Notification',
        r'File .* uploaded successfully by user .*': 'System Notification',
        r'Disk cleanup completed successfully.': 'System Notification',
        r'System reboot initiated by user .*': 'System Notification',
        r'Account with ID .* created by .*': 'User Action'
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern,log_message,re.IGNORECASE):
            return label
    return None

In [13]:
print(classify_with_regex('Backup completed successfully.'))

System Notification


In [14]:
df['regex_label']=df['log_message'].apply(classify_with_regex)
df.head()

,timestamp,source,log_message,target_label,complexity,clusters,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None


In [15]:
df_non_regex=df[df.regex_label.isna()].copy()
df_non_regex.shape

(1910, 7)

In [16]:
df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts()<=5].index.tolist()

['Workflow Error', 'Deprecation Warning']

In [17]:
df_bert=df_non_regex[~((df_non_regex.target_label=='Workflow Error') | (df_non_regex.target_label=='Deprecation Warning'))]
df_bert.shape


(1903, 7)

In [18]:
df_bert.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [19]:
#encode df_bert
filt_emb=model.encode(df_bert['log_message'].tolist())
filt_emb.shape

(1903, 384)

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
y=df_bert['target_label']
X_train,X_test,y_train,y_test=train_test_split(filt_emb,y,test_size=0.2,random_state=10)

In [21]:
y[:3]

0       HTTP Status
1    Critical Error
2    Security Alert
Name: target_label, dtype: object

In [22]:
log_model=LogisticRegression(max_iter=1000)
log_model.fit(X_train,y_train)
y_pred=log_model.predict(X_test)
print(classification_report(y_test,y_pred))

                precision    recall  f1-score   support

Critical Error       0.95      0.95      0.95        38
         Error       0.97      0.95      0.96        38
   HTTP Status       1.00      1.00      1.00       203
Resource Usage       1.00      1.00      1.00        37
Security Alert       0.98      1.00      0.99        65

      accuracy                           0.99       381
     macro avg       0.98      0.98      0.98       381
  weighted avg       0.99      0.99      0.99       381



In [23]:
import pickle
with open('bert_regr_model.pkl','wb') as f:
    pickle.dump(log_model,f)

In [38]:
import dagshub
dagshub.init(repo_owner='akinluaayomide8', repo_name='Ml_classific_proj', mlflow=True)
import mlflow

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1514e974-c7c4-49a4-9fcb-5ecebe96a593&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=15a3bdade50e6e1d54d13bb05c755c3f4f11fada786674f1d98b7e3907f414a7




Accessing as akinluaayomide8

Initialized MLflow to track repo "akinluaayomide8/Ml_classific_proj"

Repository akinluaayomide8/Ml_classific_proj initialized!

In [39]:
models=[
    (
      'LogisticRegression',
        LogisticRegression(C=1,solver='liblinear'),
        (X_train,y_train),
        (X_test,y_test)
    )]

In [40]:
reports= []
for model_name,model,train_set,test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)
reports

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


[{'0': {'precision': 0.972972972972973,
   'recall': 0.9473684210526315,
   'f1-score': 0.96,
   'support': 38.0},
  '1': {'precision': 0.9736842105263158,
   'recall': 0.9736842105263158,
   'f1-score': 0.9736842105263158,
   'support': 38.0},
  '2': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 203.0},
  '3': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 37.0},
  '4': {'precision': 0.9848484848484849,
   'recall': 1.0,
   'f1-score': 0.9923664122137404,
   'support': 65.0},
  'accuracy': 0.9921259842519685,
  'macro avg': {'precision': 0.9863011336695546,
   'recall': 0.9842105263157894,
   'f1-score': 0.9852101245480112,
   'support': 381.0},
  'weighted avg': {'precision': 0.9920948149294605,
   'recall': 0.9921259842519685,
   'f1-score': 0.9920835086453889,
   'support': 381.0}}]

In [41]:
print(f'is {mlflow.get_tracking_uri()}')

is https://dagshub.com/akinluaayomide8/Ml_classific_proj.mlflow


In [42]:
import dagshub.auth
print(dagshub.auth.get_token())

ad320b9740bb73f92fbd83ab5c6bf7c388430446


In [43]:
import os
os.environ['MLFLOW_TRACKING_USERNAME']='akinluaayomide8'
os.environ['MLFLOW_TRACKING_PASSWORD']='ad320b9740bb73f92fbd83ab5c6bf7c388430446'
os.environ['MLFLOW_TRACKING_URI']='https://dagshub.com/akinluaayomide8/Ml_classific_proj.mlflow'

In [50]:
import os
import joblib
import mlflow
import mlflow.sklearn

# Set MLflow tracking
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment('log_classifier')

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]

    # Start MLflow run
    with mlflow.start_run(run_name=model_name):

        # Log parameters
        mlflow.log_param('model_name', model_name)

        # Log metrics
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_0': report['0']['recall'],
            'recall_class_1': report['1']['recall'],
            'f1_score_class_0': report['0']['f1-score'],
            'f1_score_class_1': report['1']['f1-score'],
            'f1_score_macro_avg': report['macro avg']['f1-score']
        })

        # Save .pkl file locally
        pkl_path = f"{model_name}.pkl"
        joblib.dump(model, pkl_path)

        # Log .pkl file as artifact
        mlflow.log_artifact(pkl_path, artifact_path="models")

        # Log MLflow model (required for registry)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"Finished logging model: {model_name}")


2026/03/28 20:37:54 INFO mlflow.tracking.fluent: Experiment with name 'log_classifier' does not exist. Creating a new experiment.
2026/03/28 20:37:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 20:38:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Finished logging model: LogisticRegression
🏃 View run LogisticRegression at: https://dagshub.com/akinluaayomide8/Ml_classific_proj.mlflow/#/experiments/2/runs/ef575e447ed947039ef5e6f4ef530e87
🧪 View experiment at: https://dagshub.com/akinluaayomide8/Ml_classific_proj.mlflow/#/experiments/2
